In [1]:
import numpy as np
import pandas as pd


In [2]:
from linearmodels.panel import PanelOLS

### Data Import and Quick Clean

In [3]:
data = pd.read_csv("ELA_data_2013_2023.csv")
school_poverty = pd.read_excel('Demographic_Snapshot_2017-18_to_2021-22__Public_.xlsx', sheet_name = 'School')

In [4]:
from data_cleaning import cleaning_data, change_variable_type, create_school_level_data, merge_data_and_poverty, categorize_poverty
# clean data
school_data = cleaning_data(data) 

In [5]:
# convert num columns
cols = ['mean_scale_score','level_1_count','level_2_count','level_3_count', 'level_4_count', 'level_4_percentage', 'level_3_4_count',
        'level_1_percentage','level_2_percentage','level_3_percentage','level_4_percentage','level_3_4_percentage']
school_data = change_variable_type(school_data,cols)
# change year to object
school_data['Year'] = school_data['Year'].astype('object')

In [6]:
# create school level data, all grades all students
school_level_data = create_school_level_data(school_data)

In [7]:
# merge with poverty data
merged_data = merge_data_and_poverty(school_level_data, school_poverty)

In [8]:
# convert poverty percentage to float and categorize, round to 3 decimals
merged_data['% Poverty'] = merged_data['% Poverty'].astype(float)
merged_data['Poverty_Category'] = merged_data.apply(categorize_poverty, axis=1)
merged_data['% Poverty'] = merged_data['% Poverty'].round(3)

### Model, all grades 

In [9]:
# move to panel data format, this is necessary for PanelOLS to work properly
data2 = merged_data.set_index(['school_name','Year'])

In [10]:
# schools should have 2 years of data
data2.groupby(level=0).size().value_counts()
# why are there some schools with only 1 year of data? 3 and 4????

2    1106
1      11
4       1
3       1
Name: count, dtype: int64

In [11]:
# use only valid schools with 2 years of data for the DiD analysis, this is necessary for model to work properly 
counts = data2.groupby(level=0).size() # check count of years for each school
valid_schools = counts[counts == 2].index # filter data to include only valid schools
data2_adjust = data2.loc[valid_schools] # filter data to include only valid schools

In [12]:
# make poverty category an integer for modeling
data2_adjust['Poverty_Category'] = data2_adjust['Poverty_Category'].astype(int)

In [13]:
# create treatment, post, and did variables for DiD analysis
data2_adjust['treated'] = (data2_adjust['Poverty_Category'] == 1).astype(int)
data2_adjust['post'] = (data2_adjust.index.get_level_values('Year') == 2022).astype(int)
data2_adjust['did'] = (data2_adjust['treated'] * data2_adjust['post']).astype(int)

In [14]:
# model, looking at ALL STUDENTS, ALL GRADES, with school and year fixed effects
# cluster standard errors at the school level to account for within-school correlation over timee
mod = PanelOLS.from_formula('mean_scale_score ~  did + EntityEffects + TimeEffects', data2_adjust)
res = mod.fit(cov_type='clustered', cluster_entity=True)

# can include weights, excluded for now for baseline model
# weights=data2_adjust['number_tested']


print(res.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:       mean_scale_score   R-squared:                     3.779e-05
Estimator:                   PanelOLS   R-squared (Between):           6.194e-05
No. Observations:                2188   R-squared (Within):               0.0014
Date:                Thu, Feb 26 2026   R-squared (Overall):           6.246e-05
Time:                        22:38:17   Log-likelihood                   -4824.4
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      0.0409
Entities:                        1103   P-value                           0.8397
Avg Obs:                       1.9837   Distribution:                  F(1,1083)
Min Obs:                       1.0000                                           
Max Obs:                       2.0000   F-statistic (robust):             0.0237
                            

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


### Model, testing with Grade 3

In [25]:
school_data_3 = school_data[
    (school_data['Grade'] == '3') & 
    (school_data['Student Category'] == 'All Students') &
    (school_data['Report Category'] == 'School')
]

In [27]:
merged_data3 = merge_data_and_poverty(school_data_3, school_poverty)

In [28]:
merged_data3['% Poverty'] = merged_data3['% Poverty'].astype(float)
merged_data3['Poverty_Category'] = merged_data3.apply(categorize_poverty, axis=1)
merged_data3['% Poverty'] = merged_data3['% Poverty'].round(3)

In [29]:
data3= merged_data3.set_index(['school_name','Year'])

counts = data3.groupby(level=0).size() # check count of years for each school
valid_schools = counts[counts == 2].index # filter data to include only valid schools
data3_adjust = data3.loc[valid_schools]

data3_adjust['Poverty_Category'] = data3_adjust['Poverty_Category'].astype(int)


In [31]:
data3_adjust['treated'] = (data3_adjust['Poverty_Category'] == 1).astype(int)
data3_adjust['post'] = (data3_adjust.index.get_level_values('Year') == 2022).astype(int)
data3_adjust['did'] = (data3_adjust['treated'] * data3_adjust['post']).astype(int)

look into background of PanelOLS <br>
why use that? other alternative packages? <br>
why this package <br>


tuning <br>
levels, consistent? <br>
poverty levels - inconsistent? <br>
- prepare slide on why it's fine <br>


slide: <br>
model: what goes in, what comes out <br>
weights, how is it being weighted? 

In [35]:
mod = PanelOLS.from_formula('mean_scale_score ~  did + EntityEffects + TimeEffects', data3_adjust, weights=data3_adjust['number_tested'])
res = mod.fit(cov_type='clustered', cluster_entity=True)

# can include weights, excluded for now for baseline model
# weights=data3_adjust['number_tested']


# numbers aren't changing too much with weights



print(res.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:       mean_scale_score   R-squared:                        0.0209
Estimator:                   PanelOLS   R-squared (Between):             -0.0013
No. Observations:                1536   R-squared (Within):               0.0658
Date:                Fri, Feb 27 2026   R-squared (Overall):             -0.0013
Time:                        16:40:28   Log-likelihood                   -3617.6
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      16.248
Entities:                         771   P-value                           0.0001
Avg Obs:                       1.9922   Distribution:                   F(1,763)
Min Obs:                       1.0000                                           
Max Obs:                       2.0000   F-statistic (robust):             7.6630
                            

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


______

to do 2/26 <br>
incorporate 2018 somehow <br>
look into incorporating grades <br>
redefine treatment (maybe comparing with nyc avg is not good enough, quantiles? median?) <br>
separate grades, do individual grades, this may be bad results because incorporating all grades for the model, which relates to the next point... <br>
^^ figure out how to get consistency with grades in the school, all schooks may not have *all* of the grades 3-8, maybe for "all grades" model only use schools who have them all?


